In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
import torch
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from data_classes.decomposition import Extract_Features
import torch.utils


In [2]:
phase1_data = sio.loadmat('../data/mine_impact_data_2019.mat')
samples  = pd.DataFrame(phase1_data["x"].T)
labels  = pd.DataFrame(phase1_data["y"].T, columns=["y"])

df = pd.concat([samples, labels], axis=1, join="inner")

df = df.dropna()

In [4]:

shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df_X = shuffled_df.iloc[:, :-1]
df_Y = shuffled_df.iloc[:, -1]

data = Extract_Features(
    df_X=df_X,
    df_Y=df_Y,
    feature = "autoencoder",
    input_dim = 36000,
    latent_dim = 512,
    weight_decay = 1e-3,
    n_epochs = 5,
    batch_size = 30,
    lr =  1e-3
)

RuntimeError: MPS backend out of memory (MPS allocated: 18.10 GB, other allocations: 736.00 KB, max allowed: 18.13 GB). Tried to allocate 137.33 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [6]:
print(data.get_samples().shape)
print(data.get_labels().shape)

(3309, 512)
(3309,)


In [7]:
from sklearn.svm import SVC, LinearSVC

svc = SVC(kernel="rbf", C=100)
svc.fit(data.get_samples()[:3000], data.get_labels()[:3000])
print(svc.score(data.get_samples()[3000:], data.get_labels()[3000:]))

0.6957928802588996


In [ ]:
import models.classification as classify
import models.loops as loops

train_idx = list(range(0, 3000))
test_idx = list(range(3000,3308))

train_data = torch.utils.data.Subset(data, train_idx)
test_data = torch.utils.data.Subset(data, test_idx)

batch_size = 30

train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

model = classify.MLP(nb_hidden=64, input_dim=512, output_dim=2)

loops.train(model=model, model_path="autoencoder.pth", train_loader=train_loader, batch_size=batch_size, lr=1e-3, weight_decay=0, optim="adam", epochs=10)

loops.test(model_path="autoencoder.pth", test_loader=test_loader)

[INFO] EPOCH: 1/10
Train loss: 0.665982, Train accuracy: 0.6177
[INFO] EPOCH: 2/10
Train loss: 0.637673, Train accuracy: 0.6657
[INFO] EPOCH: 3/10
Train loss: 0.637507, Train accuracy: 0.6657
[INFO] EPOCH: 4/10
Train loss: 0.637723, Train accuracy: 0.6657
[INFO] EPOCH: 5/10
Train loss: 0.637574, Train accuracy: 0.6657
[INFO] EPOCH: 6/10
Train loss: 0.637308, Train accuracy: 0.6657
[INFO] EPOCH: 7/10
Train loss: 0.637683, Train accuracy: 0.6657
[INFO] EPOCH: 8/10
Train loss: 0.637451, Train accuracy: 0.6657
[INFO] EPOCH: 9/10
Train loss: 0.637615, Train accuracy: 0.6657
[INFO] EPOCH: 10/10
Train loss: 0.637967, Train accuracy: 0.6657
[INFO] Testing the model
Test accuracy: 0.6948
